# GRID PONDERADO
## Objetivo
Integrar las capas de la previa etapa en uN GRID , y realizar el análisis por variables.
## Inputs
- 2_Dimensiones.gpkg
## Procesos
- Carga de variables
- Creación del grid
- Carga de variables al grid
- Creación de indicadores
- Ponderado por dimensión
- Ponderado general
## Output
- 3_Grid.gpkg
    - 3_Grid.shp
    - 3_D2_Grid.shp
    - 3_D3_Grid.shp
    - 3_D4_Grid.shp
    - 3_D5_Grid.shp

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm

import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
from matplotlib import cm

import geopandas as gpd
from shapely.geometry import Polygon, Point
from dbfread import DBF
import jenkspy
import json
from jenkspy import jenks_breaks
import time
import h3
from itertools import product

In [2]:
ruta_datos = r"C:\Users\alflo\Desktop\DAL_LAB\outputs\etapa_2\2_Dimensiones.gpkg"

ruta_grid = r"C:\Users\alflo\Desktop\DAL_LAB\data_raw\Grid\GRID_ANALISIS.gpkg"

ruta_salida = r"C:\Users\alflo\Desktop\DAL_LAB\outputs\etapa_3\3_Grid_ponderado.gpkg"

# Preparativos

In [ ]:
# Configurar la fuente como predeterminada en Matplotlib
plt.rcParams['font.family'] = 'Century Gothic'
plt.rcParams['font.size'] = 12
plt.rcParams['font.weight'] = 'regular'
plt.rcParams["image.cmap"] = "tab20c"
colors = plt.get_cmap()(np.linspace(0.19, 0, 3))

In [ ]:
# Verificar columnas del DF, para ver columnas a sumar, y columnas a recalcular

# Asegúrate de que pandas muestre todas las columnas y expanda el ancho de la visualización
pd.set_option('display.max_columns', None)  # Muestra todas las columnas
#pd.set_option('display.max_colwidth', None)  # No limite el ancho de las columnas
pd.set_option('display.expand_frame_repr', False)  # Evita el corte de DataFrames grandes

#columnas

In [ ]:
def plot_colorMaps(cmap):

    fig, ax = plt.subplots(figsize=(4,0.4))
    col_map = plt.get_cmap(cmap)
    matplotlib.colorbar.ColorbarBase(ax, cmap=col_map, orientation = 'horizontal')

    plt.show()

for cmap_id in plt.colormaps():
    print(cmap_id)
    plot_colorMaps(cmap_id)

# Carga de datos

In [17]:
grid_gdf =   gpd.read_file(ruta_grid, layer = "grid_1250")

grid_gdf.rename(columns={'id':'CVE'}, inplace=True)

grid_gdf = grid_gdf.drop(columns=['left', 'top', 'right', 'bottom', 'row_index', 'col_index'])

grid_gdf.to_file(ruta_grid, layer= 'grid_1250_v2', driver="GPKG")

In [ ]:
gpd.list_layers(ruta_datos)

In [ ]:
# demanda =   gpd.read_file(ruta_datos, layer = "2_D2_mercado_demanda")
# comp_ue = gpd.read_file(ruta_datos, layer = "2_D3C1_competencia_ue")
# comp_eq =   gpd.read_file(ruta_datos, layer = "2_D3C2_competencia_eq")
# mov_l =   gpd.read_file(ruta_datos, layer = "2_D4_movilidad_lineas")
# mov_p = gpd.read_file(ruta_datos, layer = "2_D4_movilidad_puntos")
# ue =   gpd.read_file(ruta_datos, layer = "2_D5C1_ue")
# sing_ue =   gpd.read_file(ruta_datos, layer = "2_D5C1_sinergia_ue")
# sing_eq = gpd.read_file(ruta_datos, layer = "2_D5C2_sinergia_eq")
# peaton =   gpd.read_file(ruta_datos, layer = "2_D5C3_Infraestructura_peatonal_mzn")
# delitos = gpd.read_file(ruta_datos, layer = "2_D5C4_delitos")
# inundaciones =   gpd.read_file(ruta_datos, layer = "2_D5C4_inundaciones")

# Dimensión 2

## Carga de variables

In [19]:
grid_gdf =   gpd.read_file(ruta_grid, layer = "grid_1250_v2")

demanda =   gpd.read_file(ruta_datos, layer = "2_D2_mercado_demanda")

In [ ]:
demanda.columns

In [20]:
cols = ['POBFEM', 'POBMAS', 'P_60YMAS',
       'P_60YMAS_F', 'P_60YMAS_M', 'PCON_DISC', 'PCDISC_MOT', 'PCDISC_MOT2',
       'PCON_LIMI', 'PCLIM_CSB', 'PCLIM_MOT2']

demanda[cols] = demanda[cols].astype(float)

In [21]:
# manejo de variables de calidad urbana

# Asegurarnos de que ambos GeoDataFrames están en el mismo sistema de coordenadas
demanda = demanda.to_crs(grid_gdf.crs)

# Calcular intersecciones entre manzanas y hexágonos
intersecciones = gpd.overlay(demanda, grid_gdf, how='intersection')

In [22]:
# Renombrar las columnas de interés
renombres = {
    "Emp": "POBFLOT"
}
intersecciones = intersecciones.rename(columns=renombres)

In [23]:
# Actualizar lista de columnas a sumar
columnas_suma = list(renombres.values())

columnas_suma = {
       'POBTOT', 'POBFEM', 'POBMAS', 'P_60YMAS',
       'P_60YMAS_F', 'P_60YMAS_M', 'PCON_DISC', 'PCDISC_MOT', 'PCDISC_MOT2',
       'PCON_LIMI', 'PCLIM_CSB', 'PCLIM_MOT2', 'POBFLOT'
}

resultados = intersecciones.groupby("CVE").agg(
    num_manzanas=("CVEGEO", "count"),
    area_acumulada=("area_ha", "sum"),
    **{col: (col, "sum") for col in columnas_suma}
).reset_index()

In [24]:
grid_gdf = grid_gdf.merge(resultados, on='CVE', how='left')

In [25]:
grid_gdf['P_DIS'] = grid_gdf['PCON_DISC'] + grid_gdf['PCON_LIMI']
grid_gdf['P_DIS_MOT'] = grid_gdf['PCDISC_MOT'] + grid_gdf['PCLIM_CSB'] + grid_gdf['PCDISC_MOT2'] + grid_gdf['PCLIM_MOT2']
grid_gdf['POB_OBJ'] = grid_gdf['P_60YMAS'] + grid_gdf['P_DIS_MOT']
grid_gdf['POB_GRAL'] = grid_gdf['POBTOT'] + grid_gdf['POBFLOT'] - grid_gdf['POB_OBJ']

# resultados_gdf['area_ha'] = resultados_gdf.area / 10000
# resultados_gdf['DENS_GRAL'] = resultados_gdf['POB_GRAL'] / resultados_gdf['area_ha']
# resultados_gdf['DENS_OBJ'] = resultados_gdf['POB_OBJ'] / resultados_gdf['area_ha']

In [26]:
grid_gdf.columns

Index(['CVE', 'geometry', 'num_manzanas', 'area_acumulada', 'PCLIM_MOT2',
       'P_60YMAS_M', 'POBMAS', 'PCLIM_CSB', 'P_60YMAS', 'POBFEM', 'POBTOT',
       'PCDISC_MOT', 'PCON_DISC', 'POBFLOT', 'PCDISC_MOT2', 'PCON_LIMI',
       'P_60YMAS_F', 'P_DIS', 'P_DIS_MOT', 'POB_OBJ', 'POB_GRAL'],
      dtype='object')

In [27]:
grid_gdf.to_file(ruta_salida, layer= '3_D2_Mercado_demanda', driver="GPKG")

# Dimensión 3

In [28]:
grid_gdf =   gpd.read_file(ruta_grid, layer = "grid_1250_v2")

comp_ue = gpd.read_file(ruta_datos, layer = "2_D3C1_competencia_ue")
comp_eq =   gpd.read_file(ruta_datos, layer = "2_D3C2_competencia_eq")

In [29]:
# Asegurarnos de que ambos GeoDataFrames están en el mismo sistema de coordenadas
comp_ue = comp_ue.to_crs(grid_gdf.crs)
comp_eq = comp_eq.to_crs(grid_gdf.crs)

In [ ]:
comp_ue.columns

In [ ]:
comp_eq.columns

In [30]:
# Función para contar las unidades económicas y sumar empleados dentro de los polígonos
def calcular_datos(poligonos, puntos, col_suma, res_conteo, res_suma, res_prom):
    
    # Realizar un 'spatial join' para unir los puntos con los polígonos en los que caen
    join = gpd.sjoin(puntos, poligonos, how="inner", predicate='within', rsuffix='_grid')

    # Calcular el conteo de unidades económicas y la suma de empleados por polígono
    conteo = join.groupby('CVE').size()  # Cuenta las unidades económicas por CVEGEO
    suma = join.groupby('CVE')[col_suma].sum()  # Suma los empleados por CVEGEO

    # Asignar los resultados al geodataframe de poligonos usando CVEGEO como clave única
    poligonos[res_conteo] = poligonos['CVE'].map(conteo).fillna(0).astype(int)
    poligonos[res_suma] = poligonos['CVE'].map(suma).fillna(0).astype(int)
    poligonos[res_prom] = poligonos[res_suma] / poligonos[res_conteo]
    

In [31]:
# Calcular datos ue
calcular_datos(grid_gdf, comp_ue, 'Num_Emp', 'UE_c','EMP_c','TAM_UE_c')

# Calcular datos eq
calcular_datos(grid_gdf, comp_eq, 'IMP', 'EQ_c','IMP_c','IMPxEQ_c')

In [32]:
grid_gdf.columns

Index(['CVE', 'geometry', 'UE_c', 'EMP_c', 'TAM_UE_c', 'EQ_c', 'IMP_c',
       'IMPxEQ_c'],
      dtype='object')

In [33]:
grid_gdf.to_file(ruta_salida, layer= '3_D3_Competencia', driver="GPKG")

# Dimensión 5

In [34]:
grid_gdf =   gpd.read_file(ruta_grid, layer = "grid_1250_v2")

In [35]:
ue =   gpd.read_file(ruta_datos, layer = "2_D5C1_ue")
sing_ue =   gpd.read_file(ruta_datos, layer = "2_D5C1_sinergia_ue")
sing_eq = gpd.read_file(ruta_datos, layer = "2_D5C2_sinergia_eq")
peaton =   gpd.read_file(ruta_datos, layer = "2_D5C3_Infraestructura_peatonal_mzn")
delitos = gpd.read_file(ruta_datos, layer = "2_D5C4_delitos")
inundaciones =   gpd.read_file(ruta_datos, layer = "2_D5C4_inundaciones")

In [36]:
# Asegurarnos de que ambos GeoDataFrames están en el mismo sistema de coordenadas
ue = ue.to_crs(grid_gdf.crs)
sing_ue = sing_ue.to_crs(grid_gdf.crs)
sing_eq = sing_eq.to_crs(grid_gdf.crs)
peaton = peaton.to_crs(grid_gdf.crs)
delitos = delitos.to_crs(grid_gdf.crs)
inundaciones = inundaciones.to_crs(grid_gdf.crs)

## mzn

In [ ]:
peaton.columns

In [ ]:
# Calcular intersecciones entre manzanas y hexágonos
intersecciones = gpd.overlay(peaton, grid_gdf, how='intersection')

# Actualizar lista de columnas a sumar
columnas_suma = {
       'ARB', 'ALU', 'BAN'
}

resultados = intersecciones.groupby("CVE").agg(
    num_manzanas=("CVEGEO", "count"),
    # area_acumulada=("area_ha", "sum"),
    **{col: (col, "sum") for col in columnas_suma}
).reset_index()

# # Calcular valores por manzana 
for col in columnas_suma:
    resultados[f"{col}_mz"] = resultados[col] / resultados["num_manzanas"]

grid_gdf = grid_gdf.merge(resultados, on='CVE', how='left')

In [41]:
grid_gdf.columns

Index(['CVE', 'geometry', 'num_manzanas', 'ALU', 'ARB', 'BAN', 'ALU_mz',
       'ARB_mz', 'BAN_mz'],
      dtype='object')

In [49]:
inundaciones.columns

Index(['Vulnerabil', 'geometry'], dtype='object')

In [51]:
inundaciones['Vulnerabil'].unique()

array(['Muy bajo', 'Bajo', 'Medio', 'Alto', 'Muy alto'], dtype=object)

## UE y EQ

In [ ]:
ue.columns

In [ ]:
sing_ue.columns

In [ ]:
sing_eq.columns

In [ ]:
# Función para contar las unidades económicas y sumar empleados dentro de los polígonos
def calcular_datos(poligonos, puntos, col_suma, res_conteo, res_suma, res_prom):
    
    # Realizar un 'spatial join' para unir los puntos con los polígonos en los que caen
    join = gpd.sjoin(puntos, poligonos, how="inner", predicate='within', rsuffix='_grid')

    # Calcular el conteo de unidades económicas y la suma de empleados por polígono
    conteo = join.groupby('CVE').size()  # Cuenta las unidades económicas por CVEGEO
    suma = join.groupby('CVE')[col_suma].sum()  # Suma los empleados por CVEGEO

    # Asignar los resultados al geodataframe de poligonos usando CVEGEO como clave única
    poligonos[res_conteo] = poligonos['CVE'].map(conteo).fillna(0).astype(int)
    poligonos[res_suma] = poligonos['CVE'].map(suma).fillna(0).astype(int)
    poligonos[res_prom] = poligonos[res_suma] / poligonos[res_conteo]
    

# Calcular datos ue
calcular_datos(grid_gdf, ue, 'Num_Emp', 'UE','EMP','TAM_UE')

# Calcular datos ue sinergias
calcular_datos(grid_gdf, sing_ue, 'Num_Emp', 'UE_s','EMP_s','TAM_UE_s')

# Calcular datos eq sinergias
calcular_datos(grid_gdf, sing_eq, 'IMP', 'EQ_s','IMP_s','IMPxEQ_s')

In [44]:
grid_gdf.columns

Index(['CVE', 'geometry', 'num_manzanas', 'ALU', 'ARB', 'BAN', 'ALU_mz',
       'ARB_mz', 'BAN_mz', 'UE', 'EMP', 'TAM_UE', 'UE_s', 'EMP_s', 'TAM_UE_s',
       'EQ_s', 'IMP_s', 'IMPxEQ_s'],
      dtype='object')

## delitos

In [45]:
delitos.columns

Index(['fecha', 'delito', 'clave_mun', 'hora', 'Categoría', 'geometry'], dtype='object')

In [ ]:
# Función para contar las unidades económicas y sumar empleados dentro de los polígonos
def contar_por_cat(poligonos, puntos, col_cat, prefijo):

    categorias = puntos[col_cat].dropna().unique()
    
    for cat in categorias:
        #Selección de los puntos de la categoría
        puntos_cat = puntos[puntos[col_cat] == cat]
        
        # Realizar un 'spatial join' para unir los puntos con los polígonos en los que caen
        join = gpd.sjoin(puntos_cat, poligonos, how="inner", predicate='within')
    
        # Calcular el conteo 
        conteo = join.groupby('CVE').size() 

        # nombre dinámico de columna
        col_resultado = f'{prefijo}_{cat}'
        
        # asignar resultados
        poligonos[col_resultado] =poligonos['CVE'].map(conteo).fillna(0).astype(int)

# Calcular datos ue
contar_por_cat(grid_gdf, delitos, 'Categoría', 'DEL')

In [48]:
grid_gdf.columns

Index(['CVE', 'geometry', 'num_manzanas', 'ALU', 'ARB', 'BAN', 'ALU_mz',
       'ARB_mz', 'BAN_mz', 'UE', 'EMP', 'TAM_UE', 'UE_s', 'EMP_s', 'TAM_UE_s',
       'EQ_s', 'IMP_s', 'IMPxEQ_s', 'DEL_Hogar', 'DEL_Corporal', 'DEL_Bienes'],
      dtype='object')

## guardar

In [ ]:
grid_gdf.to_file(ruta_salida, layer= '3_D3_Competencia', driver="GPKG")

# Definición del usuario base

## EMA

In [ ]:
# encuestas
encuestas = pd.read_csv(r"Z:\AMIM\Alejandro\Docs\INFORME MOV\6. MIBICI\EMA\Encuesta_MiBici_2023_Todas_Las_Respuestas_Individuales_240131\CSV\Encuesta MiBici 2023.csv")
encuestas_muestra = pd.read_excel(r"Z:\AMIM\Alejandro\Docs\INFORME MOV\6. MIBICI\EMA\Encuesta_Muestra_Representativa_v1_240627.xlsx")

### grupo de edad

In [ ]:
# Eliminar la fila con id 0
encuestas = encuestas.drop(index=0, errors='ignore')

# Convertir la columna de edad a numérica
encuestas['¿Cuál es tu edad?'] = pd.to_numeric(encuestas['¿Cuál es tu edad?'], errors='coerce')

# Crear una nueva columna con los grupos quinquenales
encuestas['GRUP_QUIN'] = pd.cut(
    encuestas['¿Cuál es tu edad?'], 
    bins=range(0, 101, 5),  # Rango de 0 a 100 con intervalos de 5
    right=False,           # Intervalos abiertos a la derecha (e.g., [0-5))
    labels=[f"{i}-{i+4}" for i in range(0, 100, 5)]  # Etiquetas como '0-4', '5-9', etc.
)




# Contar la cantidad de encuestas por grupo quinquenal
grupo_counts = encuestas['GRUP_QUIN'].value_counts().sort_index()

# Crear el gráfico de barras
plt.figure(figsize=(10, 6))
plt.bar(
    grupo_counts.index, 
    grupo_counts.values, 
    color=colors)

# Personalizar el gráfico
plt.title("Distribución de encuestas por grupos quinquenales (EMA, total)", fontsize=16)
plt.xlabel("Grupo quinquenal", fontsize=12)
plt.ylabel("Cantidad de respuestas", fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()

# Mostrar el gráfico
plt.show()


In [ ]:
# Eliminar la fila con id 0
encuestas_muestra = encuestas_muestra.drop(index=0, errors='ignore')

# Convertir la columna de edad a numérica
encuestas_muestra['¿Cuál es tu edad?'] = pd.to_numeric(encuestas_muestra['¿Cuál es tu edad?'], errors='coerce')

# Crear una nueva columna con los grupos quinquenales
encuestas_muestra['GRUP_QUIN'] = pd.cut(
    encuestas_muestra['¿Cuál es tu edad?'], 
    bins=range(0, 101, 5),  # Rango de 0 a 100 con intervalos de 5
    right=False,           # Intervalos abiertos a la derecha (e.g., [0-5))
    labels=[f"{i}-{i+4}" for i in range(0, 100, 5)]  # Etiquetas como '0-4', '5-9', etc.
)




# Contar la cantidad de encuestas por grupo quinquenal
grupo_counts = encuestas_muestra['GRUP_QUIN'].value_counts().sort_index()

# Crear el gráfico de barras
plt.figure(figsize=(10, 6))
plt.bar(
    grupo_counts.index, 
    grupo_counts.values, 
    color=colors)

# Personalizar el gráfico
plt.title("Distribución de encuestas por grupos quinquenales (EMA, muestra)", fontsize=16)
plt.xlabel("Grupo quinquenal", fontsize=12)
plt.ylabel("Cantidad de respuestas", fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()

# Mostrar el gráfico
plt.show()


### Ingresos mensuales

In [ ]:
# Normalizar los nombres de las categorías de ingreso
encuestas['¿Cuál es tu ingreso mensual aproximado? (personal)Pesos mexicanos'] = (
    encuestas['¿Cuál es tu ingreso mensual aproximado? (personal)Pesos mexicanos']
    .str.replace('$', '', regex=False)  # Quitar el símbolo de $
    .str.strip()  # Eliminar espacios adicionales
)

# Mapeo para agrupar y ordenar las categorías de ingreso
mapa_ingresos = {
    'Menos de 5,000': 'Menos de 5,000',
    'Entre 5,000 y 14,999': 'Entre 5,000 y 14,999',
    'Entre 15,000 y 24,999': 'Entre 15,000 y 24,999',
    'Entre 25,000 y 34,999': 'Entre 25,000 y 34,999',
    'Entre 35,000 y 44,999': 'Entre 35,000 y 44,999',
    'Entre 45,000 y 59,999': 'Más de 45,000',
    'Entre 60,000 y 74,999': 'Más de 45,000',
    'Entre 75,000 y 99,000': 'Más de 45,000',
    'Más de 99,000': 'Más de 45,000',
    'Prefiero no responder': 'Prefiero no responder'
}

# Aplicar el mapeo
encuestas['Ingreso Agrupado'] = encuestas['¿Cuál es tu ingreso mensual aproximado? (personal)Pesos mexicanos'].map(mapa_ingresos)

# Ordenar las categorías de forma ascendente
orden_ingresos = [
    'Menos de 5,000',
    'Entre 5,000 y 14,999',
    'Entre 15,000 y 24,999',
    'Entre 25,000 y 34,999',
    'Entre 35,000 y 44,999',
    'Más de 45,000'
]

encuestas['Ingreso Agrupado'] = pd.Categorical(
    encuestas['Ingreso Agrupado'], 
    categories=orden_ingresos, 
    ordered=True
)

#1 Agrupar por quinquenios e ingresos, y contar la frecuencia
frecuencias_ingresos = (
    encuestas.groupby(['GRUP_QUIN', 'Ingreso Agrupado'])
    .size()
    .unstack(fill_value=0)
)

# 2. Filtrar filas correspondientes a los grupos quinquenales de interés
grupos_objetivo = ['20-24', '25-29', '30-34']
frecuencias_ingresos = frecuencias_ingresos.loc[grupos_objetivo]

# Calcular el total general de personas en los grupos quinquenales seleccionados
total_general = frecuencias_ingresos.sum().sum()

# Calcular los porcentajes relativos al total general
porcentajes_ingresos = (frecuencias_ingresos / total_general) * 100

# Configurar estilo de gráfico
ax=porcentajes_ingresos.T.plot(kind='bar', figsize=(10, 6), stacked=True, color=colors)

# Añadir etiquetas a las barras
for container in ax.containers:
    for bar in container:
        # Obtener la altura de cada barra
        height = bar.get_height()
        if height > 0:  # Solo etiquetar barras con valores positivos
            ax.text(
                bar.get_x() + bar.get_width() / 2,  # Posición x centrada
                bar.get_y() + height / 2,          # Posición y centrada
                f'{height:.1f}%',                  # Etiqueta con formato
                ha='center', va='center',         # Centrar etiqueta
                fontsize=8, color='white'        # Personalización del texto
            )

plt.title('Distribución de ingresos aproximados por grupo quinquenal (EMA, total)', fontsize=16)
plt.xlabel('Ingreso mensual aproximado (Pesos mexicanos)', labelpad = 18, fontsize=14)
plt.ylabel('Porcentaje del total general', labelpad = 18, fontsize=14)
plt.legend(title='Grupo quinquenal', title_fontsize=13)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()

# Mostrar la gráfica
plt.show()


In [ ]:
# Normalizar los nombres de las categorías de ingreso
encuestas_muestra['¿Cuál es tu ingreso mensual aproximado? (personal)Pesos mexicanos'] = (
    encuestas_muestra['¿Cuál es tu ingreso mensual aproximado? (personal)Pesos mexicanos']
    .str.replace('$', '', regex=False)  # Quitar el símbolo de $
    .str.strip()  # Eliminar espacios adicionales
)

# Mapeo para agrupar y ordenar las categorías de ingreso
mapa_ingresos = {
    'Menos de 5,000': 'Menos de 5,000',
    'Entre 5,000 y 14,999': 'Entre 5,000 y 14,999',
    'Entre 15,000 y 24,999': 'Entre 15,000 y 24,999',
    'Entre 25,000 y 34,999': 'Entre 25,000 y 34,999',
    'Entre 35,000 y 44,999': 'Entre 35,000 y 44,999',
    'Entre 45,000 y 59,999': 'Más de 45,000',
    'Entre 60,000 y 74,999': 'Más de 45,000',
    'Entre 75,000 y 99,000': 'Más de 45,000',
    'Más de 99,000': 'Más de 45,000',
    'Prefiero no responder': 'Prefiero no responder'
}

# Aplicar el mapeo
encuestas_muestra['Ingreso Agrupado'] = encuestas_muestra['¿Cuál es tu ingreso mensual aproximado? (personal)Pesos mexicanos'].map(mapa_ingresos)

# Ordenar las categorías de forma ascendente
orden_ingresos = [
    'Menos de 5,000',
    'Entre 5,000 y 14,999',
    'Entre 15,000 y 24,999',
    'Entre 25,000 y 34,999',
    'Entre 35,000 y 44,999',
    'Más de 45,000'
]

encuestas_muestra['Ingreso Agrupado'] = pd.Categorical(
    encuestas_muestra['Ingreso Agrupado'], 
    categories=orden_ingresos, 
    ordered=True
)

#1 Agrupar por quinquenios e ingresos, y contar la frecuencia
frecuencias_ingresos = (
    encuestas_muestra.groupby(['GRUP_QUIN', 'Ingreso Agrupado'])
    .size()
    .unstack(fill_value=0)
)

# 2. Filtrar filas correspondientes a los grupos quinquenales de interés
grupos_objetivo = ['20-24', '25-29', '30-34']
frecuencias_ingresos = frecuencias_ingresos.loc[grupos_objetivo]

# Calcular el total general de personas en los grupos quinquenales seleccionados
total_general = frecuencias_ingresos.sum().sum()

# Calcular los porcentajes relativos al total general
porcentajes_ingresos = (frecuencias_ingresos / total_general) * 100

# Configurar estilo de gráfico
ax=porcentajes_ingresos.T.plot(kind='bar', figsize=(10, 6), stacked=True, color=colors)

# Añadir etiquetas a las barras
for container in ax.containers:
    for bar in container:
        # Obtener la altura de cada barra
        height = bar.get_height()
        if height > 0:  # Solo etiquetar barras con valores positivos
            ax.text(
                bar.get_x() + bar.get_width() / 2,  # Posición x centrada
                bar.get_y() + height / 2,          # Posición y centrada
                f'{height:.1f}%',                  # Etiqueta con formato
                ha='center', va='center',         # Centrar etiqueta
                fontsize=8, color='white'        # Personalización del texto
            )

plt.title('Distribución de ingresos aproximados por grupo quinquenal (EMA, muestra)', fontsize=16)
plt.xlabel('Ingreso mensual aproximado (Pesos mexicanos)', labelpad = 18, fontsize=14)
plt.ylabel('Porcentaje del total general', labelpad = 18, fontsize=14)
plt.legend(title='Grupo quinquenal', title_fontsize=13)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()

# Mostrar la gráfica
plt.show()


## EMCA

In [ ]:
# Configuración inicial
excel_file = r"Z:\AMIM\Alejandro\Docs\INFORME MOV\5. EMCA\Encuestas\CIC_2023_limpio.xlsx"

# Leer archivo Excel y extraer las columnas necesarias
emca = pd.read_excel(excel_file)

### grupo de edad

In [ ]:
# # Convertir la columna de edad a numérica
# emca['¿Cuál es tu edad?'] = pd.to_numeric(emca['¿Cuál es tu edad?'], errors='coerce')

# Crear una nueva columna con los grupos quinquenales
emca['GRUP_QUIN'] = pd.cut(
    emca['Q6'], 
    bins=range(0, 101, 5),  # Rango de 0 a 100 con intervalos de 5
    right=False,           # Intervalos abiertos a la derecha (e.g., [0-5))
    labels=[f"{i}-{i+4}" for i in range(0, 100, 5)]  # Etiquetas como '0-4', '5-9', etc.
)


# Contar la cantidad de encuestas por grupo quinquenal
grupo_counts = emca['GRUP_QUIN'].value_counts().sort_index()

# Crear el gráfico de barras
plt.figure(figsize=(10, 6))
plt.bar(
    grupo_counts.index, 
    grupo_counts.values, 
    color=colors)

# Personalizar el gráfico
plt.title("Distribución de encuestas por grupos quinquenales (EMCA)", fontsize=16)
plt.xlabel("Grupo quinquenal", fontsize=12)
plt.ylabel("Cantidad de respuestas", fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()

# Mostrar el gráfico
plt.show()

### Ingresos mensuales

In [ ]:
# Normalizar los nombres de las categorías de ingreso
emca['Q11'] = (
    emca['Q11']
    .str.replace('$', '', regex=False)  # Quitar el símbolo de $
    .str.strip()  # Eliminar espacios adicionales
)

# Mapeo para agrupar y ordenar las categorías de ingreso
mapa_ingresos = {
    'Menos de 5,000': 'Menos de 5,000',
    'Entre 5,000 y 14,999': 'Entre 5,000 y 14,999',
    'Entre 15,000 y 24,999': 'Entre 15,000 y 24,999',
    'Entre 25,000 y 34,999': 'Entre 25,000 y 34,999',
    'Entre 35,000 y 44,999': 'Entre 35,000 y 44,999',
    'Entre 45,000 y 59,999': 'Más de 45,000',
    'Entre 60,000 y 74,999': 'Más de 45,000',
    'Entre 75,000 y 99,000': 'Más de 45,000',
    'Más de 99,000': 'Más de 45,000',
    'Prefiero no responder': 'Prefiero no responder'
}

# Aplicar el mapeo
emca['Ingreso Agrupado'] = emca['Q11'].map(mapa_ingresos)

# Ordenar las categorías de forma ascendente
orden_ingresos = [
    'Menos de 5,000',
    'Entre 5,000 y 14,999',
    'Entre 15,000 y 24,999',
    'Entre 25,000 y 34,999',
    'Entre 35,000 y 44,999',
    'Más de 45,000'
]

emca['Ingreso Agrupado'] = pd.Categorical(
    emca['Ingreso Agrupado'], 
    categories=orden_ingresos, 
    ordered=True
)

#1 Agrupar por quinquenios e ingresos, y contar la frecuencia
frecuencias_ingresos = (
    emca.groupby(['GRUP_QUIN', 'Ingreso Agrupado'])
    .size()
    .unstack(fill_value=0)
)

# 2. Filtrar filas correspondientes a los grupos quinquenales de interés
grupos_objetivo = ['20-24', '25-29', '30-34']
frecuencias_ingresos = frecuencias_ingresos.loc[grupos_objetivo]

# Calcular el total general de personas en los grupos quinquenales seleccionados
total_general = frecuencias_ingresos.sum().sum()

# Calcular los porcentajes relativos al total general
porcentajes_ingresos = (frecuencias_ingresos / total_general) * 100

# Configurar estilo de gráfico
ax=porcentajes_ingresos.T.plot(kind='bar', figsize=(10, 6), stacked=True, color=colors)

# Añadir etiquetas a las barras
for container in ax.containers:
    for bar in container:
        # Obtener la altura de cada barra
        height = bar.get_height()
        if height > 0:  # Solo etiquetar barras con valores positivos
            ax.text(
                bar.get_x() + bar.get_width() / 2,  # Posición x centrada
                bar.get_y() + height / 2,          # Posición y centrada
                f'{height:.1f}%',                  # Etiqueta con formato
                ha='center', va='center',         # Centrar etiqueta
                fontsize=8, color='white'        # Personalización del texto
            )

plt.title('Distribución de ingresos aproximados por grupo quinquenal (EMCA)', fontsize=16)
plt.xlabel('Ingreso mensual aproximado (Pesos mexicanos)', labelpad = 18, fontsize=14)
plt.ylabel('Porcentaje del total general', labelpad = 18, fontsize=14)
plt.legend(title='Grupo quinquenal', title_fontsize=13)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()

# Mostrar la gráfica
plt.show()


## censo loc

In [ ]:

#RUTAS ENTRADA - ALEJANDRO (SSD)
ITER_2020_ruta = r"Z:\VOCES\DATA\2020\ITER\ITER_NALCSV20.csv"
ITER_2010_ruta = r"Z:\VOCES\DATA\2010\ITER\ITER_NALDBF10.dbf"

# ruta shp base
SHP_LOC_NAC_ruta = r"Z:\VOCES\DATA\SHP\mg_2020_integrado\conjunto_de_datos\00l.shp"

#ruta gpkg fin
MAPA_TIPO_ruta = r"Z:\AMIM\Alejandro\Docs\MT\Loc_Censo.gpkg"

# Ruta del archivo shp de polígonos
polygons_path = r"Z:\VOCES\DATA\PCU_2018_SHP\PCUS_2018.shp"

#rutas claves metro
CVE_METRO_ruta = r"Z:\VOCES\DATA\Claves_Metro.csv"

#rutas denue
ruta_denue = r"Z:\VOCES\DATA\Tratados\NAC\DENUE.gpkg"

### base de datos

In [ ]:
# PASO DOS - Preparación de información censal

# Función para cargar datos de archivo DBF y convertirlos a DataFrame
def load_dbf_to_dataframe(file_path):
    table = DBF(file_path)
    return pd.DataFrame(iter(table))

# Cargar archivos en un DataFrame
ITER_2020_df = pd.read_csv(ITER_2020_ruta, dtype={'ENTIDAD': str, 'MUN': str, 'LOC': str, 'AGEB': str, 'MZA': str})
ITER_2010_df = load_dbf_to_dataframe(ITER_2010_ruta)

# Homologación de ID rural - CVEGEO
def homologate_rural_id(row):
    return f"{row['ENTIDAD']}{row['MUN']}{row['LOC']}"

ITER_2020_df['CVEGEO'] = ITER_2020_df.apply(homologate_rural_id, axis=1)
ITER_2010_df['CVEGEO'] = ITER_2010_df.apply(homologate_rural_id, axis=1)

# reemplazar * con 1
def replace(dataframe):
    return dataframe.replace('*', '1')

ITER_2020_df = replace(ITER_2020_df)
ITER_2010_df = replace(ITER_2010_df)


# Leer el archivo de polígonos
polygons_gdf = gpd.read_file(polygons_path)[['geometry', 'CALIF_CLAS']]

# Define el CRS proyectado que deseas utilizar
CRS_PROYECTADO = 'EPSG:6372' 

# Leer shp base y convertir a gpkg
SHP_LOC_gdf = gpd.read_file(SHP_LOC_NAC_ruta)

# Proyecta el GeoDataFrame a un CRS proyectado
SHP_LOC_gdf = SHP_LOC_gdf.to_crs(CRS_PROYECTADO)
polygons_gdf = polygons_gdf.to_crs(CRS_PROYECTADO)

# CREAR GPKG BASE DESDE gdf DE MANZANAS
SHP_LOC_gdf.to_file(MAPA_TIPO_ruta, layer='Mapa_Localidades', driver='GPKG')

# Leer gpkg base
MAPA_TIPO_gdf = gpd.read_file(MAPA_TIPO_ruta, layer='Mapa_Localidades')

# Filtrar entidades rurales que intersectan con los polígonos
MAPA_TIPO_gdf_INTERSECT = gpd.sjoin(MAPA_TIPO_gdf, polygons_gdf, how='left', predicate='intersects')

# Eliminar duplicados basados en el identificador de SHP_LOC_gdf
MAPA_TIPO_gdf_INTERSECT = MAPA_TIPO_gdf_INTERSECT.drop_duplicates(subset=SHP_LOC_gdf.columns.tolist())

# Guardar la capa concatenada en el GeoPackage base
MAPA_TIPO_gdf_INTERSECT.to_file(MAPA_TIPO_ruta, layer='Mapa_Localidades', driver='GPKG', overwrite=True)

In [ ]:
# Leer gpkg base
MAPA_TIPO_gdf = gpd.read_file(MAPA_TIPO_ruta, layer='Mapa_Localidades')

# Filtrar censos
ITER_2020_df_F = ITER_2020_df[['CVEGEO', 'POBTOT','P_20A24', 'P_20A24_F', 'P_20A24_M', 'P_25A29', 'P_25A29_F', 'P_25A29_M', 'P_30A34', 'P_30A34_F','P_30A34_M', 'VIVTOT', 'VIVPAR_DES']]
#ITER_2010_df_F = ITER_2010_df[['CVEGEO', 'POBTOT','P_20A24', 'P_20A24_F', 'P_20A24_M', 'P_25A29', 'P_25A29_F', 'P_25A29_M', 'P_30A34', 'P_30A34_F','P_30A34_M', 'VIVTOT', 'VIVPAR_DES']]

# Renombrar columnas de ITER_2020_df antes del merge para evitar conflictos
ITER_2020_df_renamed = ITER_2020_df_F.rename(columns={
    'POBTOT'    : 'POPTOT2020',
    'P_20A24'   :   'P_20A242020'   ,
    'P_20A24_F' : 'P_20A24_F2020' , 
    'P_20A24_M' : 'P_20A24_M2020' , 
    'P_25A29'   :   'P_25A292020'   , 
    'P_25A29_F' : 'P_25A29_F2020' , 
    'P_25A29_M' : 'P_25A29_M2020' , 
    'P_30A34'   :   'P_30A342020'   , 
    'P_30A34_F' : 'P_30A34_F2020' ,
    'P_30A34_M' : 'P_30A34_M2020' ,
    'VIVTOT'    : 'VIVTOT2020',
    'VIVPAR_DES': 'VIVPAR_DES2020'
})

# ITER_2020_df_renamed = ITER_2020_df_F.rename(columns={
#     'POBTOT': 'POPTOT2010',
#     'P_20A24'  :   'P_20A242010'   ,
#     'P_20A24_F': 'P_20A24_F2010' , 
#     'P_20A24_M': 'P_20A24_M2010' , 
#     'P_25A29'  :   'P_25A292010'   , 
#     'P_25A29_F': 'P_25A29_F2010' , 
#     'P_25A29_M': 'P_25A29_M2010' , 
#     'P_30A34'  :   'P_30A342010'   , 
#     'P_30A34_F': 'P_30A34_F2010' ,
#     'P_30A34_M': 'P_30A34_M2010' ,
#     'VIVTOT': 'VIVTOT2010',
#     'VIVPAR_DES': 'VIVPAR_DES2010'
# })

MAPA_TIPO_gdf['CVEGEO'] = MAPA_TIPO_gdf['CVEGEO'].astype(str)
ITER_2020_df_renamed['CVEGEO'] = ITER_2020_df_renamed['CVEGEO'].astype(str)
#ITER_2010_df_renamed['CVEGEO'] = ITER_2010_df_renamed['CVEGEO'].astype(str)

# Unir campos de los DataFrames al GeoPackage base por CVEGEO
MAPA_TIPO_gdf1 = MAPA_TIPO_gdf.merge(
    ITER_2020_df_renamed[['CVEGEO', 'POPTOT2020','P_20A242020','P_20A24_F2020' ,
                          'P_20A24_M2020' , 'P_25A292020','P_25A29_F2020' ,'P_25A29_M2020' , 
                          'P_30A342020' ,'P_30A34_F2020','P_30A34_M2020' ,
                          'VIVTOT2020','VIVPAR_DES2020']],
    on='CVEGEO', how='left')

# MAPA_TIPO_gdf2 = MAPA_TIPO_gdf1.merge(
#     ITER_2010_df_renamed[['CVEGEO', 'POPTOT2010','P_20A242010','P_20A24_F2010' ,
#                           'P_20A24_M2010' ,  'P_25A292010' ,'P_25A29_F2010' ,'P_25A29_M2010' ,
#                           'P_30A342010' ,'P_30A34_F2010' ,'P_30A34_M2010' ,
#                           'VIVTOT2010','VIVPAR_DES2010']],
#     on='CVEGEO', how='left')

# Reemplazar * con 1
def replace(dataframe):
    return dataframe.replace('*', '1')

# Limpiar *
MAPA_TIPO_gdf2_A = replace(MAPA_TIPO_gdf1)

# Reemplazar N/D
MAPA_TIPO_gdf2_B = MAPA_TIPO_gdf2_A.replace('N/D', np.nan)

# Actualizar valores sin información de "null" a 0
MAPA_TIPO_gdf2_C = MAPA_TIPO_gdf2_B.fillna(0)

# Convertir campos a enteros
fields_to_convert = [
    'POPTOT2020','P_20A242020','P_20A24_F2020' ,
    'P_20A24_M2020' , 'P_25A292020','P_25A29_F2020' ,'P_25A29_M2020' , 
    'P_30A342020' ,'P_30A34_F2020','P_30A34_M2020' ,
    'VIVTOT2020','VIVPAR_DES2020']
for field in fields_to_convert:
    MAPA_TIPO_gdf2_C[field] = MAPA_TIPO_gdf2_C[field].astype(int)

# # Reordenar columnas
# ordered_columns = [
#     'CVEGEO',  'POPTOT2020', 'P_20A242020', 
#     'P_20A24_F2020', 'P_20A24_M2020',
#     'P_25A292020', 'P_25A29_F2020',
#     'P_25A29_M2020', 'P_30A342020',
#     'P_30A34_F2020', 'P_30A34_M2020', 
#     'VIVTOT2020', 'VIVPAR_DES2020'
# ] + [col for col in MAPA_TIPO_gdf1.columns if col not in [
#     'CVEGEO',  'POPTOT2020', 'P_20A242020', 
#     'P_20A24_F2020', 'P_20A24_M2020',
#     'P_25A292020', 'P_25A29_F2020',
#     'P_25A29_M2020', 'P_30A342020',
#     'P_30A34_F2020', 'P_30A34_M2020', 
#     'VIVTOT2020', 'VIVPAR_DES2020'
# ]]


# MAPA_TIPO_gdf2_C = MAPA_TIPO_gdf2_C[ordered_columns]

# Guardar el GeoPackage base actualizado
MAPA_TIPO_gdf2_C.to_file(MAPA_TIPO_ruta, layer='BASE_DATOS_NAC_LOC', driver="GPKG")

In [ ]:
# Leer gpkg base
MAPA_TIPO_gdf = gpd.read_file(MAPA_TIPO_ruta, layer='BASE_DATOS_NAC_LOC')

# Cargar la segunda capa del segundo GeoPackage (individualmente)
MAPA_TIPO_gdf2_C_FIL = MAPA_TIPO_gdf[MAPA_TIPO_gdf['CVE_ENT'] == '14']

# Guardar la capa filtrada del segundo GeoPackage
MAPA_TIPO_gdf2_C_FIL.to_file(MAPA_TIPO_ruta, layer='Mapa_Localidades_JAL', driver="GPKG")


In [ ]:
#AGRUPAR Y ASIGNAR FILTRO Y PROM POR ZM

#Subir archivo con CVE_METRO
CVE_METRO_df = pd.read_csv(CVE_METRO_ruta, dtype={'CVE_MET': str, 'CVEMUN':str})

# Leer la capa existente del GeoPackage base
MAPA_TIPO_gdf = gpd.read_file(MAPA_TIPO_ruta, layer='Mapa_Localidades_JAL')

#Crear campo CVEMUN
def homologate_municipio_id(row):
    return f"{row['CVE_ENT']}{row['CVE_MUN']}"

MAPA_TIPO_gdf['CVEMUN'] = MAPA_TIPO_gdf.apply(homologate_municipio_id, axis=1)

#Añadir claves a GPKG
MAPA_TIPO_gdf = MAPA_TIPO_gdf.merge(CVE_METRO_df[['CVEMUN', 'CVE_MET', 'NOM_MET', 'TIPO_MET','TIPO_MUN','NOM_MUN']], on='CVEMUN', how='left')

# Guardar el GeoPackage base actualizado
MAPA_TIPO_gdf.to_file(MAPA_TIPO_ruta, layer= 'Mapa_Localidades_JAL', driver="GPKG")

### Obtencion de datos 

In [ ]:
# --- 1. Agrupación de datos por municipio y cálculo de totales ---

# Lista de municipios de interés
municipios = ['002','039', '044', '051','070','097', '098', '101', '120', '124']

# Leer la capa existente del GeoPackage base
MAPA_TIPO_gdf = gpd.read_file(MAPA_TIPO_ruta, layer='Mapa_Localidades_JAL')

# Filtrar el DataFrame solo con los municipios de interés
AMG_gdf = MAPA_TIPO_gdf[MAPA_TIPO_gdf['CVE_MUN'].isin(municipios)]

# Seleccionar las columnas de interés (totales y por género)
columnas_interes = [
    'CVE_MUN',  'NOM_MUN','P_20A242020', 'P_25A292020', 'P_30A342020',
    'P_20A24_F2020', 'P_20A24_M2020', 'P_25A29_F2020', 'P_25A29_M2020',
    'P_30A34_F2020', 'P_30A34_M2020'
]

# Crear un DataFrame filtrado con las columnas necesarias
df_interes = AMG_gdf[columnas_interes].copy()

# Crear columna de total por grupo quinquenal (20-34 años)
df_interes['TOTAL_20A34'] = (
    df_interes['P_20A242020'] + df_interes['P_25A292020'] + df_interes['P_30A342020']
)

# Agrupar por 'CVE_MUN' y sumar solo las columnas numéricas
tabla_totales = df_interes.groupby('CVE_MUN', as_index=False).sum(numeric_only=True)

# # Conservar 'NOM_MUN' usando el primer valor no nulo en cada grupo
tabla_totales['NOM_MUN'] = df_interes.groupby('CVE_MUN')['NOM_MUN'].first().reset_index(drop=True)

# Agregar fila de total general
fila_total = tabla_totales.drop(columns='CVE_MUN').sum(numeric_only=True)
fila_total['CVE_MUN'] = 'TOTAL'
tabla_totales = pd.concat([tabla_totales, pd.DataFrame([fila_total])], ignore_index=True)

# Mostrar la tabla resultante
print("Tabla de Totales por Municipio y Grupo Quinquenal:")
tabla_totales

In [ ]:
# --- Datos de entrada ---
grupos_quinquenales = ['P_20A242020', 'P_25A292020', 'P_30A342020']
grupos_femenino = ['P_20A24_F2020', 'P_25A29_F2020', 'P_30A34_F2020']
grupos_masculino = ['P_20A24_M2020', 'P_25A29_M2020', 'P_30A34_M2020']

# Calcular totales por género para cada grupo quinquenal
data_genero = pd.DataFrame({
    'Grupo Quinquenal': ['20-24', '25-29', '30-34'],
    'Femenino': [df_interes[col].sum() for col in grupos_femenino],
    'Masculino': [df_interes[col].sum() for col in grupos_masculino],
})

# Calcular porcentajes
data_genero['Total'] = data_genero['Femenino'] + data_genero['Masculino']
data_genero['% Femenino'] = (data_genero['Femenino'] / data_genero['Total'] * 100).round(1)
data_genero['% Masculino'] = (data_genero['Masculino'] / data_genero['Total'] * 100).round(1)

# --- Crear la gráfica ---
fig, ax = plt.subplots(figsize=(10, 6))

# Barras apiladas
bar_fem = ax.bar(data_genero['Grupo Quinquenal'], data_genero['Femenino'], 
                label='Femenino', color=colors[1])
bar_masc = ax.bar(data_genero['Grupo Quinquenal'], data_genero['Masculino'], 
                  bottom=data_genero['Femenino'], label='Masculino', color=colors[2])

# Agregar etiquetas con porcentajes
for i, (fem, masc) in enumerate(zip(bar_fem, bar_masc)):
    # Etiqueta Femenino
    ax.text(fem.get_x() + fem.get_width() / 2, fem.get_height() / 2,
            f"{data_genero['% Femenino'][i]}%", ha='center', va='center', color='white', fontsize=10)
    # Etiqueta Masculino
    ax.text(masc.get_x() + masc.get_width() / 2, fem.get_height() + masc.get_height() / 2,
            f"{data_genero['% Masculino'][i]}%", ha='center', va='center', color='white', fontsize=10)

# Personalización
ax.set_title('Distribución de Hombres y Mujeres por Grupo Quinquenal (INEGI)', fontsize=14)
ax.set_xlabel('Grupo Quinquenal', fontsize=12)
ax.set_ylabel('Total de Población', fontsize=12)
ax.legend(title='Género', loc='upper right')

# Ajustar apariencia
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


# Tratamiento de bases de datos

## Tratamiento de fuentes

In [ ]:
# RUTAS ENTRADA - ALEJANDRO (SSD)

#RUTAS CENSOS
MZNA_2020_ruta = r"Z:\VOCES\DATA\Tratados\NAC\MZNA_2020_NAC.csv"
ITER_2020_ruta = r"Z:\VOCES\DATA\Tratados\NAC\ITER_2020_NAC.csv"

# ruta shp base
SHP_LOC_NAC_ruta = r"Z:\VOCES\DATA\SHP\NAC\00l.shp"
SHP_MZN_NAC_ruta = r"Z:\VOCES\DATA\Tratados\NAC\MZNyLOC_NAC.gpkg"

#ruta gpkg fin
MAPA_TIPO_ruta = r"Z:\AMIM\Alejandro\Docs\MT\Mzn_Censo.gpkg"

# Ruta del archivo shp de polígonos de contención urbana
polygons_path = r"Z:\VOCES\DATA\PCU_2018_SHP\PCUS_2018.shp"

#CLAVES METRO
CVE_METRO_ruta = r"Z:\VOCES\DATA\Claves_Metro.csv"

#Claves denue
DENUE_ruta = r"Z:\VOCES\DATA\Tratados\NAC\DENUE.gpkg"

#rutas (ALEJANDRO - SSD)
equipamientos_gpkg = r"Z:\AMIM\Alejandro\Docs\MT\Fuentes\Equipamientos_NAC_2020.gpkg"

Censo_Urbn_ruta = r"Z:\AMIM\Alejandro\Docs\DATA\Censos\Censo2020_CEU_jal_csv\TI_MANZANA_EU_14.csv"

In [ ]:
##########CREAR GPKG BASE

# Leer el archivo de polígonos
polygons_gdf = gpd.read_file(polygons_path)[['geometry', 'CALIF_CLAS']]

# Define el CRS proyectado que deseas utilizar
CRS_PROYECTADO = 'EPSG:6372' 

# Leer shp base
SHP_MZN_gdf= gpd.read_file(SHP_MZN_NAC_ruta, layer='MZN')
SHP_LOC_gdf = gpd.read_file(SHP_LOC_NAC_ruta)

# Proyecta el GeoDataFrame a un CRS proyectado
SHP_MZN_gdf = SHP_MZN_gdf.to_crs(CRS_PROYECTADO)
SHP_LOC_gdf = SHP_LOC_gdf.to_crs(CRS_PROYECTADO)
polygons_gdf = polygons_gdf.to_crs(CRS_PROYECTADO)

# Selección de ITER rural
SHP_LOC_gdf_RURAL = SHP_LOC_gdf[SHP_LOC_gdf['AMBITO'] == 'Rural']

# Filtrar entidades rurales que intersectan con los polígonos
SHP_LOC_gdf_RURAL_INTERSECT = gpd.sjoin(SHP_LOC_gdf_RURAL, polygons_gdf, how='inner', predicate='intersects')

# Selección de MZN Urbana
SHP_MZN_gdf_URB = SHP_MZN_gdf[SHP_MZN_gdf['AMBITO'] == 'Urbana']

# Concatenar ITER rural a MZN urbana
gdf_concatenated = gpd.GeoDataFrame(pd.concat([SHP_MZN_gdf_URB, SHP_LOC_gdf_RURAL_INTERSECT], ignore_index=True))

# Guardar la capa concatenada en el GeoPackage base
gdf_concatenated.to_file(MAPA_TIPO_ruta, layer='Mapa_MT', driver='GPKG')

In [ ]:
# Leer gpkg base
MAPA_TIPO_gdf = gpd.read_file(MAPA_TIPO_ruta, layer='Mapa_MT')

# EXTRAER JALISCO
MAPA_TIPO_gdf_JAL = MAPA_TIPO_gdf[MAPA_TIPO_gdf['CVE_ENT'] == '14']

# Lista de municipios AMG
municipios = ['002','039', '044', '051','070','097', '098', '101', '120', '124']

# Filtrar el DataFrame solo con los municipios de interés
MAPA_TIPO_gdf_AMG = MAPA_TIPO_gdf[
    (MAPA_TIPO_gdf['CVE_MUN'].isin(municipios)) & 
    (MAPA_TIPO_gdf['CVE_ENT'] == '14')
]

# Guardar la capa filtrada del segundo GeoPackage
MAPA_TIPO_gdf_JAL.to_file(MAPA_TIPO_ruta, layer='Mapa_MT_JAL', driver="GPKG")
MAPA_TIPO_gdf_AMG.to_file(MAPA_TIPO_ruta, layer='Mapa_MT_AMG', driver="GPKG")

In [ ]:
# Carga el GeoDataFrame de manzanas urbanas
gdf_manzanas = gpd.read_file(MAPA_TIPO_ruta, layer='Mapa_MT_AMG')

#Checar por duplicados
# Detectar duplicados basados en el identificador CVEGEO
duplicados = gdf_manzanas[gdf_manzanas.duplicated(subset="CVEGEO", keep=False)]

if not duplicados.empty:
    print("Se encontraron manzanas duplicadas:")
    print(duplicados.shape)

    # Eliminar duplicados, conservando el primero que aparece
    gdf_manzanas = gdf_manzanas.drop_duplicates(subset="CVEGEO", keep="first")
    print(f"Se han eliminado los duplicados. Total de registros ahora: {len(gdf_manzanas)}")
else:
    print("No se encontraron manzanas duplicadas.")

# Guardar el GeoPackage base actualizado
gdf_manzanas.to_file(MAPA_TIPO_ruta, layer='Mapa_MT_AMG', driver="GPKG")

In [ ]:
# Leer gpkg base
MAPA_TIPO_gdf = gpd.read_file(MAPA_TIPO_ruta, layer='Mapa_MT_AMG')

# LEER INFORMACIÓN CENSAL
MZNA_2020_df = pd.read_csv(MZNA_2020_ruta, dtype=str)
ITER_2020_df = pd.read_csv(ITER_2020_ruta, dtype=str)

# Filtrar censos
MZNA_2020_df_F = MZNA_2020_df[['CVEGEO', 'POBTOT', 'VIVTOT', 'P_18YMAS', 'P_18YMAS_F', 'P_18YMAS_M', 'P_60YMAS', 'P_60YMAS_F', 'P_60YMAS_M', 'VIVPAR_DES', 'PEA', 'PE_INAC', 'POCUPADA', 'PDESOCUP', 'VPH_AUTOM', 'VPH_MOTO', 'VPH_BICI']]
ITER_2020_df_F = ITER_2020_df[['CVEGEO', 'POBTOT', 'VIVTOT', 'P_18YMAS', 'P_18YMAS_F', 'P_18YMAS_M', 'P_60YMAS', 'P_60YMAS_F', 'P_60YMAS_M', 'VIVPAR_DES', 'PEA', 'PE_INAC', 'POCUPADA', 'PDESOCUP', 'VPH_AUTOM', 'VPH_MOTO', 'VPH_BICI']]
# Renombrar columnas de ITER_2020_df antes del merge para evitar conflictos

# Crear un diccionario base con las columnas seleccionadas
columnas_seleccionadas = {
    'POBTOT': 'POBTOT',
    'VIVTOT': 'VIVTOT',
    'VIVPAR_DES': 'VIVPAR_DES',
    'PEA': 'PEA',
    'PE_INAC': 'PE_INAC',
    'POCUPADA': 'POCUPADA',
    'PDESOCUP': 'PDESOCUP',
    'VPH_AUTOM': 'VPH_AUTOM',
    'VPH_MOTO': 'VPH_MOTO',
    'VPH_BICI': 'VPH_BICI',
    'P_18YMAS': 'P_18YMAS',
    'P_18YMAS_F': 'P_18YMAS_F',
    'P_18YMAS_M': 'P_18YMAS_M',
    'P_60YMAS': 'P_60YMAS',
    'P_60YMAS_F': 'P_60YMAS_F',
    'P_60YMAS_M': 'P_60YMAS_M'
}

# Función para renombrar columnas con un prefijo
def renombrar_columnas(dataframe, prefijo):
    return dataframe.rename(columns={col: f"{col}_{prefijo}" for col in columnas_seleccionadas})

# Renombrar las columnas en ambos DataFrames
MZNA_2020_df_renamed = renombrar_columnas(MZNA_2020_df_F, "MZNA")
ITER_2020_df_renamed = renombrar_columnas(ITER_2020_df_F, "ITER")

MAPA_TIPO_gdf['CVEGEO'] = MAPA_TIPO_gdf['CVEGEO'].astype(str)
MZNA_2020_df_renamed['CVEGEO'] = MZNA_2020_df_renamed['CVEGEO'].astype(str)
ITER_2020_df_renamed['CVEGEO'] = ITER_2020_df_renamed['CVEGEO'].astype(str)

# Obtener dinámicamente las columnas renombradas para cada DataFrame
columnas_mzna = ['CVEGEO'] + [f"{col}_MZNA" for col in columnas_seleccionadas]
columnas_iter = ['CVEGEO'] + [f"{col}_ITER" for col in columnas_seleccionadas]

# Realizar los merges
MAPA_TIPO_gdf1 = MAPA_TIPO_gdf.merge(
    MZNA_2020_df_renamed[columnas_mzna],
    on='CVEGEO', 
    how='left'
)

MAPA_TIPO_gdf2 = MAPA_TIPO_gdf1.merge(
    ITER_2020_df_renamed[columnas_iter],
    on='CVEGEO', 
    how='left'
)

# Reemplazar * con 1
def replace(dataframe):
    return dataframe.replace('*', '1')

# Limpiar *
MAPA_TIPO_gdf4_A = replace(MAPA_TIPO_gdf2)

# Reemplazar N/D
MAPA_TIPO_gdf4_B = MAPA_TIPO_gdf4_A.replace('N/D', np.nan)

# Actualizar valores sin información de "null" a 0
MAPA_TIPO_gdf4_C = MAPA_TIPO_gdf4_B.fillna(0)


# Crear una lista de los campos combinados que deben convertirse a enteros
fields_to_convert = list(columnas_seleccionadas.keys())

# Convertir los campos combinados a enteros y manejar valores NaN o inf
for field in fields_to_convert:
    mzna_col = f"{field}_MZNA"
    iter_col = f"{field}_ITER"
    
    # Asegurarnos de que las columnas existan antes de realizar la operación
    if mzna_col in MAPA_TIPO_gdf4_C.columns and iter_col in MAPA_TIPO_gdf4_C.columns:
        MAPA_TIPO_gdf4_C[field] = (
            MAPA_TIPO_gdf4_C[mzna_col].fillna(0).replace([np.inf, -np.inf], 0).astype(float) + 
            MAPA_TIPO_gdf4_C[iter_col].fillna(0).replace([np.inf, -np.inf], 0).astype(float)
        ).round(0).astype(int)

# Eliminar las columnas individuales después de combinar
columns_to_drop = [f"{col}_MZNA" for col in columnas_seleccionadas] + \
                  [f"{col}_ITER" for col in columnas_seleccionadas]

MAPA_TIPO_gdf4_C = MAPA_TIPO_gdf4_C.drop(columns=columns_to_drop)

# Guardar el GeoPackage base actualizado
MAPA_TIPO_gdf4_C.to_file(MAPA_TIPO_ruta, layer='Mapa_MT_AMG', driver="GPKG")

In [ ]:
#CALCULAR CAMPOS UE_20 Y Emp_20

# Cargar los GeoDataFrames de las manzanas y las unidades económicas
MAPA_TIPO_gdf = gpd.read_file(MAPA_TIPO_ruta, layer='Mapa_MT_AMG')

SHP_DENUE_2020_gdf = gpd.read_file(DENUE_ruta, layer='2020')

# Verificar que ambos datasets tienen el mismo sistema de coordenadas
if MAPA_TIPO_gdf.crs != SHP_DENUE_2020_gdf.crs:
    SHP_DENUE_2020_gdf = SHP_DENUE_2020_gdf.to_crs(MAPA_TIPO_gdf.crs)

# Función para contar las unidades económicas y sumar empleados dentro de los polígonos
def calcular_datos(localidades, ue_data, col_ue, col_emp):
    # Realizar un buffer de 1 metro
    localidades_buffer = localidades.copy()
    localidades_buffer['geometry'] = localidades_buffer.geometry.buffer(1)
    
    # Realizar un 'spatial join' para unir los puntos con los polígonos en los que caen
    join = gpd.sjoin(ue_data, localidades_buffer, how="inner", predicate='within', rsuffix='_localidad')

    # Calcular el conteo de unidades económicas y la suma de empleados por polígono
    conteo_ue = join.groupby('CVEGEO').size()  # Cuenta las unidades económicas por CVEGEO
    suma_emp = join.groupby('CVEGEO')['num_emp'].sum()  # Suma los empleados por CVEGEO

    # Asignar los resultados al geodataframe de localidades usando CVEGEO como clave única
    localidades[col_ue] = localidades['CVEGEO'].map(conteo_ue).fillna(0).astype(int)
    localidades[col_emp] = localidades['CVEGEO'].map(suma_emp).fillna(0).astype(int)

# Calcular datos
calcular_datos(MAPA_TIPO_gdf, SHP_DENUE_2020_gdf, 'UE_20', 'Emp_20')

# Guardar el GeoPackage base actualizado
MAPA_TIPO_gdf.to_file(MAPA_TIPO_ruta, layer='Mapa_MT_AMG', driver="GPKG")

In [ ]:
#limpiar equipamientos

gdf_equipamientos = gpd.read_file(equipamientos_gpkg, layer='Educación')

gdf_equipamientos['TIPO'].unique()

gdf_equipamientos = gdf_equipamientos[gdf_equipamientos['TIPO'] == 'Superior']

#Guardar la capa resultante en un archivo .gpkg
gdf_equipamientos.to_file(equipamientos_gpkg, layer='Educación_Sup', driver="GPKG")

In [ ]:
#3.1: Cargar el archivo de manzanas

# Cargar el archivo de manzanas
gdf_manzanas = gpd.read_file(MAPA_TIPO_ruta, layer='Mapa_MT_AMG')

# Crear un buffer de 2 metros alrededor de las manzanas
gdf_manzanas['buffer_2m'] = gdf_manzanas.geometry.buffer(2)

# Establecer el buffer como la geometría activa
gdf_manzanas = gdf_manzanas.set_geometry('buffer_2m')

# Inicializar las columnas para contar equipamientos en cada categoría
gdf_manzanas['num_edu'] = 0
gdf_manzanas['num_sal'] = 0
gdf_manzanas['num_rec'] = 0
gdf_manzanas['num_ser'] = 0

# Lista de categorías y sus respectivas columnas de conteo
categorias = {
    'Educación_Sup': 'num_edu',
    'Salud': 'num_sal',
    'Recreación': 'num_rec',
    'Servicios': 'num_ser'
}

In [ ]:
# Cargar cada capa de equipamientos desde el geopackage y contar los elementos por manzana
for categoria, columna_conteo in categorias.items():
    # Cargar la capa correspondiente desde el geopackage
    gdf_categoria = gpd.read_file(equipamientos_gpkg, layer=categoria)
    
    # Realizar un join espacial entre los buffers de manzanas y los equipamientos de la categoría
    joined = gpd.sjoin(gdf_manzanas[['buffer_2m']], gdf_categoria, how='left', predicate='contains')
    
    # Contar los equipamientos de la categoría por cada manzana
    gdf_manzanas[columna_conteo] = joined.groupby(joined.index)['index_right'].count()

    print(f"Conteo completado para la categoría: {categoria}")

#borrar columna del buffer
gdf_manzanas = gdf_manzanas.drop(columns=['buffer_2m'])

# Establecer el buffer como la geometría activa
gdf_manzanas = gdf_manzanas.set_geometry('geometry')

#Guardar la capa resultante en un archivo .gpkg
gdf_manzanas.to_file(MAPA_TIPO_ruta, layer='Mapa_MT_AMG', driver="GPKG")

In [ ]:
# Leer gpkg base
MAPA_TIPO_gdf = gpd.read_file(MAPA_TIPO_ruta, layer='Mapa_MT_AMG')

# LEER INFORMACIÓN CENSAL
MZNA_2020_df = pd.read_csv(Censo_Urbn_ruta, dtype=str)

In [ ]:
# Filtrar censos
MZNA_2020_df_F = MZNA_2020_df[['ENT', 'MUN', 'LOC', 'AGEB', 'MZA', 'ARBOLES_C', 'ALUMPUB_C', 'BANQUETA_C']]

In [ ]:
# Lista de columnas a procesar
campos = ['ARBOLES_C', 'ALUMPUB_C', 'BANQUETA_C']

# Diccionario de reemplazo
replacement_map = {
    "1": 100,
    "2": 40,
    "3": 0,
    "8": 0,
    "9": 0
}

# Aplicar reemplazos
for field in campos:
    if field in MZNA_2020_df_F.columns:
        MZNA_2020_df_F[field] = (
            MZNA_2020_df_F[field]
            .replace(replacement_map)   # Reemplazar según el mapa
            .apply(lambda x: x if x in replacement_map.values() else np.nan)  # Convertir el resto a NaN
        )


In [ ]:
#Crear CVEGEO
MZNA_2020_df_F['CVEGEO'] = MZNA_2020_df_F['ENT'] + MZNA_2020_df_F['MUN'] + MZNA_2020_df_F['LOC'] + MZNA_2020_df_F['AGEB'] + MZNA_2020_df_F['MZA']

# Renombrar columnas de ITER_2020_df antes del merge para evitar conflictos
MAPA_TIPO_gdf['CVEGEO'] = MAPA_TIPO_gdf['CVEGEO'].astype(str)
MZNA_2020_df_F['CVEGEO'] = MZNA_2020_df_F['CVEGEO'].astype(str)

# Obtener dinámicamente las columnas renombradas para cada DataFrame
columnas_mzna = ['CVEGEO', 'ARBOLES_C', 'ALUMPUB_C', 'BANQUETA_C']

# Realizar los merges
MAPA_TIPO_gdf1 = MAPA_TIPO_gdf.merge(
    MZNA_2020_df_F[columnas_mzna],
    on='CVEGEO', 
    how='left'
)

In [ ]:
# Guardar el GeoPackage base actualizado
MAPA_TIPO_gdf1.to_file(MAPA_TIPO_ruta, layer='Mapa_MT_AMG', driver="GPKG")

## creación de indicadores

In [ ]:
# Leer la capa existente del GeoPackage base
MAPA_TIPO_gdf = gpd.read_file(MAPA_TIPO_ruta, layer='Mapa_MT_AMG')

In [ ]:
# creación de columnas

MAPA_TIPO_gdf['area_ha'] = MAPA_TIPO_gdf.area / 10000
MAPA_TIPO_gdf['DENS_POP20'] = MAPA_TIPO_gdf['POBTOT'] / MAPA_TIPO_gdf['area_ha']
MAPA_TIPO_gdf['DENS_VIV20'] = MAPA_TIPO_gdf['VIVTOT'] / MAPA_TIPO_gdf['area_ha']

MAPA_TIPO_gdf['DENS_PEA'] = MAPA_TIPO_gdf['PEA'] / MAPA_TIPO_gdf['area_ha']
MAPA_TIPO_gdf['POR_PEA'] = MAPA_TIPO_gdf['PEA'] / MAPA_TIPO_gdf['POBTOT']

MAPA_TIPO_gdf['DENS_AUTO'] = MAPA_TIPO_gdf['VPH_AUTOM'] / MAPA_TIPO_gdf['area_ha']
MAPA_TIPO_gdf['POR_AUTO'] = MAPA_TIPO_gdf['VPH_AUTOM'] / MAPA_TIPO_gdf['VIVTOT']
MAPA_TIPO_gdf['DENS_MOTO'] = MAPA_TIPO_gdf['VPH_MOTO'] / MAPA_TIPO_gdf['area_ha']
MAPA_TIPO_gdf['POR_MOTO'] = MAPA_TIPO_gdf['VPH_MOTO'] / MAPA_TIPO_gdf['VIVTOT']
MAPA_TIPO_gdf['DENS_BICI'] = MAPA_TIPO_gdf['VPH_BICI'] / MAPA_TIPO_gdf['area_ha']
MAPA_TIPO_gdf['POR_BICI'] = MAPA_TIPO_gdf['VPH_BICI'] / MAPA_TIPO_gdf['VIVTOT']

MAPA_TIPO_gdf['P_18_59'] = MAPA_TIPO_gdf['P_18YMAS'] -  MAPA_TIPO_gdf['P_60YMAS']
MAPA_TIPO_gdf['P_18_59_F'] = MAPA_TIPO_gdf['P_18YMAS_F'] - MAPA_TIPO_gdf['P_60YMAS_F']
MAPA_TIPO_gdf['P_18_59_M'] = MAPA_TIPO_gdf['P_18YMAS_M'] - MAPA_TIPO_gdf['P_60YMAS_M']

MAPA_TIPO_gdf['DENS_UEC20'] = MAPA_TIPO_gdf['UE_20'] / MAPA_TIPO_gdf['area_ha']
MAPA_TIPO_gdf['DENS_EMP20'] = MAPA_TIPO_gdf['Emp_20'] / MAPA_TIPO_gdf['area_ha']

# #categorías por predominancia de bicicletas
# MAPA_TIPO_gdf['PREDOM'] = np.where((MAPA_TIPO_gdf['POR_AUTO'] > MAPA_TIPO_gdf['POR_BICI'])
#                                    & (MAPA_TIPO_gdf['POR_AUTO'] > MAPA_TIPO_gdf['POR_MOTO']), 'AUTO', 
#                                     np.where((MAPA_TIPO_gdf['POR_MOTO'] > MAPA_TIPO_gdf['POR_AUTO'])
#                                              & (MAPA_TIPO_gdf['POR_MOTO'] > MAPA_TIPO_gdf['POR_BICI']), 'MOTO',
#                                              np.where((MAPA_TIPO_gdf['POR_BICI'] >= MAPA_TIPO_gdf['POR_AUTO'])
#                                                       & (MAPA_TIPO_gdf['POR_BICI'] >= MAPA_TIPO_gdf['POR_MOTO']), 
#                                                       'BICI','=')))



# Clasificación por predominancia y nuevas combinaciones
def clasificar_predominio(row):
    POR_AUTO = row['POR_AUTO']
    POR_MOTO = row['POR_MOTO']
    POR_BICI = row['POR_BICI']
    
    # Casos específicos
    if POR_AUTO > POR_BICI and POR_AUTO > POR_MOTO:
        return 'AUTO'
    elif POR_AUTO == POR_MOTO > POR_BICI:
        return 'AUTO'
    elif POR_MOTO > POR_AUTO and POR_MOTO > POR_BICI:
        return 'MOTO'
    elif POR_BICI > POR_AUTO and POR_BICI > POR_MOTO:
        return 'BICI'
    elif POR_AUTO == POR_BICI > POR_MOTO:
        return 'BICI'
    elif POR_MOTO == POR_BICI > POR_AUTO:
        return 'BICI'
    elif POR_AUTO == POR_BICI == POR_MOTO and POR_AUTO != 0:
        return 'IGUALDAD'
    elif POR_AUTO == POR_BICI == POR_MOTO == 0:
        return 'SIN VEHÍCULOS'
    else:
        return 'N/A'

# Aplicar la función a cada fila del GeoDataFrame
MAPA_TIPO_gdf['PREDOM'] = MAPA_TIPO_gdf.apply(clasificar_predominio, axis=1)


In [ ]:
# Guardar el GeoPackage base actualizado
MAPA_TIPO_gdf.to_file(MAPA_TIPO_ruta, layer= 'Mapa_MT_AMG', driver="GPKG")

# Unidad de medida y modelado

In [ ]:
#ruta gpkg fin
MAPA_TIPO_ruta = r"Z:\AMIM\Alejandro\Docs\MT\Mzn_Censo.gpkg"

Grid_ruta = r"Z:\AMIM\Alejandro\Docs\MT\GRID_ANALISIS.gpkg"

DENUE_ruta = r"Z:\VOCES\DATA\Tratados\NAC\DENUE.gpkg"

#rutas (ALEJANDRO - SSD)
equipamientos_gpkg = r"Z:\AMIM\Alejandro\Docs\MT\Fuentes\Equipamientos_NAC_2020.gpkg"
ruta_eq_amg =r"Z:\AMIM\Alejandro\Docs\MT\Fuentes\equipamientos_amg.gpkg"

ruta_mibici = r"Z:\AMIM\Alejandro\Docs\MT\Fuentes\estaciones_mibici_25.gpkg"
ruta_cv = r"C:\Users\AMIM GTAC\Desktop\CV_2024\CV_2024.shp"
ruta_tpm = r"Z:\AMIM\Alejandro\Docs\MT\Fuentes\TRANSPORTE P MASIVO.gpkg"

# trabajar en .to_crs('EPSG:6372')

## Creación del grid

In [ ]:
# Carga el GeoDataFrame de manzanas urbanas
MAPA_TIPO_gdf = gpd.read_file(MAPA_TIPO_ruta, layer='Mapa_MT_AMG')

### Calidad urbana

In [ ]:
# manejo de variables de calidad urbana

# Carga el GeoDataFrame de manzanas urbanas
grid_gdf = gpd.read_file(Grid_ruta, layer='grid_1250')

# Asegurarnos de que ambos GeoDataFrames están en el mismo sistema de coordenadas
MAPA_TIPO_gdf = MAPA_TIPO_gdf.to_crs(grid_gdf.crs)

# Calcular intersecciones entre manzanas y hexágonos
intersecciones = gpd.overlay(MAPA_TIPO_gdf, grid_gdf, how='intersection')

# Renombrar las columnas de interés
renombres = {
    "ARBOLES_C": "ARB", "ALUMPUB_C": "ALU", "BANQUETA_C": "BAN",
    "UE_20": "UE", "Emp_20": "EMP", "num_edu": "EQ_EDU",
    "num_sal": "EQ_SAL", "num_rec": "EQ_REC", "num_ser": "EQ_SER"
}
intersecciones = intersecciones.rename(columns=renombres)

# Actualizar lista de columnas a sumar
columnas_suma = list(renombres.values())

resultados = intersecciones.groupby("id").agg(
    num_manzanas=("CVEGEO", "count"),
    area_acumulada=("area_ha", "sum"),
    **{col: (col, "sum") for col in columnas_suma}
).reset_index()

# Calcular densidades por manzana y por hectárea
for col in columnas_suma:
    resultados[f"{col}_MZ"] = resultados[col] / resultados["num_manzanas"]
    resultados[f"{col}_HA"] = resultados[col] / resultados["area_acumulada"]

# Hacer el merge inverso para traer solo la columna geometry
resultados = pd.merge(resultados, grid_gdf[["id", "geometry"]], on="id", how="left")

# Convertir el DataFrame resultante a un GeoDataFrame
resultados_gdf = gpd.GeoDataFrame(resultados, geometry="geometry", crs=grid_gdf.crs)

# Exportar como GeoPackage
resultados_gdf.to_file(Grid_ruta, layer='grid_1250_resultados', driver="GPKG")

In [ ]:
# reconteo de empleos y UE por GRID

# Carga el GeoDataFrame de ciclovías, mibici y estaciones
SHP_DENUE_2020_gdf = gpd.read_file(DENUE_ruta, layer='2020')

#Carga del grid
resultados_gdf = gpd.read_file(Grid_ruta, layer='grid_1250_resultados')

# Verificar que ambos datasets tienen el mismo sistema de coordenadas
if resultados_gdf.crs != SHP_DENUE_2020_gdf.crs:
    SHP_DENUE_2020_gdf = SHP_DENUE_2020_gdf.to_crs(resultados_gdf.crs)

SHP_DENUE_AMG_gdf = gpd.clip(SHP_DENUE_2020_gdf, resultados_gdf)

resultados_gdf.rename(columns={'id':'CVE'}, inplace=True)

# Función para contar las unidades económicas y sumar empleados dentro de los polígonos
def calcular_datos(localidades, ue_data, col_ue, col_emp):
    
    # Realizar un 'spatial join' para unir los puntos con los polígonos en los que caen
    join = gpd.sjoin(ue_data, localidades, how="inner", predicate='within', rsuffix='_grid')

    # Calcular el conteo de unidades económicas y la suma de empleados por polígono
    conteo_ue = join.groupby('CVE').size()  # Cuenta las unidades económicas por CVEGEO
    suma_emp = join.groupby('CVE')['num_emp'].sum()  # Suma los empleados por CVEGEO

    # Asignar los resultados al geodataframe de localidades usando CVEGEO como clave única
    localidades[col_ue] = localidades['CVE'].map(conteo_ue).fillna(0).astype(int)
    localidades[col_emp] = localidades['CVE'].map(suma_emp).fillna(0).astype(int)

# Calcular datos
calcular_datos(resultados_gdf, SHP_DENUE_AMG_gdf, 'UE_g', 'EMP_g')

# Guardar el GeoPackage base actualizado
resultados_gdf.to_file(Grid_ruta, layer='grid_1250_resultados', driver="GPKG")

In [ ]:
######### Ponderación de equipamientos por área urbana

#manzanas: crear campo eq_total

# Cargar el archivo de manzanas
gdf_manzanas = gpd.read_file(MAPA_TIPO_ruta, layer='Mapa_MT_AMG')

gdf_manzanas['EQ_TOT'] = gdf_manzanas['num_edu'] + gdf_manzanas['num_sal'] + gdf_manzanas['num_rec'] + gdf_manzanas['num_ser']

# Lista de categorías y sus respectivas columnas de conteo
categorias = {
    'Educación_Sup': 'num_edu',
    'Salud': 'num_sal',
    'Recreación': 'num_rec',
    'Servicios': 'num_ser'
}

# Lista de municipios AMG
municipios = ['002','039', '044', '051','070','097', '098', '101', '120', '124']

# Cargar cada capa de equipamientos desde el geopackage y unir con manzanas
for categoria, columna_conteo in categorias.items():
    # Cargar la capa correspondiente desde el geopackage
    gdf_categoria = gpd.read_file(equipamientos_gpkg, layer=categoria)

    # Filtrar AMG
    gdf_categoria_AMG = gdf_categoria[
        (gdf_categoria['CVE_MUN'].isin(municipios)) & 
        (gdf_categoria['CVE_ENT'] == '14')
    ]

    # Unión espacial con gdf_manzanas para traer 'area_ha' y 'EQ_TOT'
    gdf_categoria_AMG = gdf_categoria_AMG.sjoin(gdf_manzanas[['geometry', 'area_ha', 'EQ_TOT']], how="left")

    #creación columna tamaño eq
    gdf_categoria_AMG['TAM_EQ'] = gdf_categoria_AMG['area_ha']/gdf_categoria_AMG['EQ_TOT']

    # Guardar la capa resultante en un archivo .gpkg
    gdf_categoria_AMG.to_file(ruta_eq_amg, layer=f'{categoria}_AMG', driver="GPKG")

    print(f"Procesamiento completado para la categoría: {categoria}")

#Carga del grid
resultados_gdf = gpd.read_file(Grid_ruta, layer='grid_1250_resultados')

# Lista de categorías y sus respectivas columnas de conteo
categorias = {
    'Educación_Sup': 'EQ_EDU_2',
    'Salud': 'EQ_SAL_2',
    'Recreación': 'EQ_REC_2',
    'Servicios': 'EQ_SER_2'
}

# Cargar cada capa de equipamientos desde el geopackage y calcular estadísticas por manzana
for categoria, columna_conteo in categorias.items():
    # Cargar la capa correspondiente desde el geopackage
    gdf_categoria = gpd.read_file(ruta_eq_amg, layer=f'{categoria}_AMG')

    #Eliminar la columna index_right
    gdf_categoria = gdf_categoria.drop(columns=['index_right'])
    
    # Realizar un join espacial entre las manzanas y los equipamientos de la categoría
    joined = gpd.sjoin(resultados_gdf, gdf_categoria, how='left', predicate='contains')

    # Calcular conteo de equipamientos por manzana
    conteo = joined.groupby(joined.index)['index_right'].count()

    # Calcular sumatoria de la columna TAM_EQ por manzana (ignorar nulos)
    sumatoria_tam_eq = joined.groupby(joined.index)['TAM_EQ'].sum()  # min_count=1 evita que NaN se convierta en 0

    # Asignar los valores a resultados_gdf
    resultados_gdf[columna_conteo] = conteo
    resultados_gdf[f'{columna_conteo}_ha'] = sumatoria_tam_eq  # Nueva columna con la sumatoria de TAM_EQ

    print(f"Conteo y sumatoria completados para la categoría: {categoria}")

# Guardar el GeoPackage base actualizado
resultados_gdf.to_file(Grid_ruta, layer='grid_1250_resultados', driver="GPKG")

### Conectividad

In [ ]:
# manejo de variables de accesibilidad

#Conteo simple por grid

# Carga el GeoDataFrame de ciclovías, mibici y estaciones
MiBici_gdf = gpd.read_file(ruta_mibici, layer='estaciones_bien_copy')
CV_gdf = gpd.read_file(ruta_cv)
TPM_gdf = gpd.read_file(ruta_tpm, layer='TRANSPORTE P MASIVO')

#Carga del grid
resultados_gdf = gpd.read_file(Grid_ruta, layer='grid_1250_resultados')

# Contar puntos de MiBici_gdf por hexágono
MiBici_gdf = MiBici_gdf.to_crs(resultados_gdf.crs)
resultados_gdf["EST_MB"] = resultados_gdf.geometry.apply(
    lambda hex_geom: MiBici_gdf[MiBici_gdf.geometry.within(hex_geom)].shape[0]
)

# Calcular kilómetros de ciclovías (CV_gdf) por hexágono
CV_gdf = CV_gdf.to_crs(resultados_gdf.crs)
resultados_gdf["KM_CV"] = resultados_gdf.geometry.apply(
    lambda hex_geom: CV_gdf.intersection(hex_geom).length.sum() / 1000  # Convertir de metros a kilómetros
)

# Contar puntos de TPM_gdf por hexágono
TPM_gdf = TPM_gdf.to_crs(resultados_gdf.crs)
resultados_gdf["EST_TP"] = resultados_gdf.geometry.apply(
    lambda hex_geom: TPM_gdf[TPM_gdf.geometry.within(hex_geom)].shape[0]
)

# Exportar como GeoPackage
resultados_gdf.to_file(Grid_ruta, layer='grid_1250_resultados', driver="GPKG")


# oooooooo
# buffer a cada una - radio de influencia
# categorias de influencia por manzana
# ponderado por grid en base a manzanas

## Ponderación y resultado

In [ ]:
#Carga del grid
resultados_gdf = gpd.read_file(Grid_ruta, layer='grid_1250_resultados')

In [ ]:
resultados_gdf.columns

In [ ]:
def normalize_columns(grid, columns):
    # Verificar y eliminar columnas duplicadas
    # resultados_gdf = resultados_gdf.loc[:, ~resultados_gdf.columns.duplicated()]
    
    for column in columns:
        if column in grid.columns:
            max_value = grid[column].max()
            if max_value != 0:  # Evitar división por cero
                norm_column = f'{column}_N'
                grid[norm_column] = (grid[column] / max_value)  * 100
            else:
                grid[f'norm_{column}'] = 0
    return grid

# Lista de las columnas que deseas normalizar
columns_to_normalize = ['UE_HA', 'EMP_HA',
                        'EQ_EDU', 'EQ_SAL', 'EQ_REC', 'EQ_SER','EQ_EDU_HA','EQ_SAL_HA','EQ_REC_HA','EQ_SER_HA',
                        'UE_g', 'EMP_g', 
                        'EQ_EDU_2', 'EQ_EDU_2_ha', 'EQ_SAL_2', 'EQ_SAL_2_ha', 'EQ_REC_2', 'EQ_REC_2_ha', 'EQ_SER_2', 'EQ_SER_2_ha',
                         'EST_MB', 'KM_CV', 'EST_TP']

# Normalizar las columnas
resultados_gdf = normalize_columns(resultados_gdf, columns_to_normalize)

In [ ]:
# Calcular Equipamientos combinados
resultados_gdf['EQ_N'] = (
    resultados_gdf['EQ_EDU_2_N'] * 0.50 + 
    resultados_gdf['EQ_SAL_2_N'] * 0.21 +
    resultados_gdf['EQ_REC_2_N'] * 0.13 + 
    resultados_gdf['EQ_SER_2_N'] * 0.16   
)

# Calcular Equipamientos combinados
resultados_gdf['EQ_ha_N'] = (
    resultados_gdf['EQ_EDU_2_ha_N'] * 0.50 + 
    resultados_gdf['EQ_SAL_2_ha_N'] * 0.21 +
    resultados_gdf['EQ_REC_2_ha_N'] * 0.13 + 
    resultados_gdf['EQ_SER_2_ha_N'] * 0.16   
)


In [ ]:
# Calcular Infraestructura
resultados_gdf['INFRA'] = (
    resultados_gdf['BAN_MZ'] * 0.424 + 
    resultados_gdf['ARB_MZ'] * 0.210 + 
    resultados_gdf['ALU_MZ'] * 0.365 
)

# Calcular Necesidades de Traslado
resultados_gdf['NEC_TRAS_D'] = (
    resultados_gdf['EMP_g_N']  * 0.701 + 
    resultados_gdf['EQ_EDU_2_N']* 0.299
)

# Calcular Necesidades de Traslado
resultados_gdf['NEC_TRAS_E'] = (
    resultados_gdf['EQ_SAL_2_N'] * 0.416 +
    resultados_gdf['EQ_REC_2_N'] * 0.257 +
    resultados_gdf['EQ_SER_2_N'] * 0.327  
)

# Calcular 1. Calidad urbana
resultados_gdf['CAL_URB'] = (
    resultados_gdf['INFRA'] * 0.305 + 
    resultados_gdf['NEC_TRAS_D']* 0.533 +
    resultados_gdf['NEC_TRAS_E']* 0.162
)

# Calcular Movilidad Activa
resultados_gdf['MOV_ACT'] = (
    resultados_gdf['EST_MB_N'] * 0.327 + 
    resultados_gdf['KM_CV_N']  * 0.673
)

# Calcular 2. Conectividad
resultados_gdf['CONECT'] = (
    resultados_gdf['MOV_ACT'] * 0.432 +  
    resultados_gdf['EST_TP_N'] * 0.568
)

# Calcular 3. Potencial de densificación
resultados_gdf['POT_DENS'] = (
    resultados_gdf['CAL_URB'] * 0.627 + 
    resultados_gdf['CONECT']  * 0.373 
)

In [ ]:
# Exportar como GeoPackage
resultados_gdf.to_file(Grid_ruta, layer='grid_1250_resultados', driver="GPKG")